# CNN Encoder - ResNet-18 Backbone

Notebook for CNN encoder implementation

## 1. Imports

In [8]:
import pandas as pd
import torch
from torch.utils.data import DataLoader, random_split
import torchvision.transforms as T
from PIL import Image

print('True')

True


In [9]:

# reload cleaned csv
df = pd.read_csv('/kaggle/input/datasets/manjushwarkhairkar/cleaned-fashion-tech-myntra-dataset-with-features/fashion_preprocessed.csv')
df = df.dropna(subset=['proc_img_path', 'proc_sketch_path']).reset_index(drop=True)
print(f"Loaded {len(df)} rows")

Loaded 14311 rows


## 2. Building frozen ResNet-18 encoder

In [3]:
class SketchEncoder(nn.Module):

    def __init__(self):
        super().__init__()

        # load pretrained ResNet-18
        resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

        # drop the avgpool and final fc layer
        # we want the spatial feature map [512, 7, 7]
        self.backbone = nn.Sequential(*list(resnet.children())[:-2])

        # freeze all parameters — we never update these weights
        for param in self.backbone.parameters():
            param.requires_grad = False

        # 1x1 conv to adapt sketch (1 channel) → (3 channel)
        # ResNet expects 3 channel input (RGB)
        self.input_adapter = nn.Conv2d(1, 3, kernel_size=1, bias=False)

    def forward(self, sketch):
        # sketch: [B, 1, 224, 224]
        x = self.input_adapter(sketch)   # → [B, 3, 224, 224]
        x = self.backbone(x)             # → [B, 512, 7, 7]
        return x

print('True')

True


## 3. Instantiate and verify

In [4]:
encoder = SketchEncoder()
print(encoder)

# count trainable vs frozen params
total    = sum(p.numel() for p in encoder.parameters())
trainable = sum(p.numel() for p in encoder.parameters() if p.requires_grad)
frozen   = total - trainable

print(f"\nTotal params    : {total:,}")
print(f"Trainable params: {trainable:,}   ← only input_adapter")
print(f"Frozen params   : {frozen:,}   ← full ResNet-18 backbone")

print('True')

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 141MB/s]

SketchEncoder(
  (backbone): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (4): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stat

## 4. Required Pydataset Class

In [ ]:
# ── Cell 1: rebuild everything from saved CSV ──
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader, Subset
from PIL import Image
import torchvision.transforms as T

# reload cleaned csv
df = pd.read_csv('/kaggle/working/fashion_preprocessed_clean.csv')
df = df.dropna(subset=['proc_img_path', 'proc_sketch_path']).reset_index(drop=True)
print(f"Loaded {len(df)} rows")

# transforms — same as dataset.ipynb
img_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std =[0.229, 0.224, 0.225])
])

sketch_transform = T.Compose([
    T.Resize((224, 224)),
    T.Grayscale(num_output_channels=1),
    T.ToTensor(),
    T.Normalize(mean=[0.5], std=[0.5])
])

# MyntraSketchDataset class
class MyntraSketchDataset(Dataset):
    def __init__(self, dataframe, img_tfm=None, sketch_tfm=None):
        self.df         = dataframe.reset_index(drop=True)
        self.img_tfm    = img_tfm
        self.sketch_tfm = sketch_tfm

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row    = self.df.iloc[idx]
        img    = Image.open(row['proc_img_path']).convert('RGB')
        sketch = Image.open(row['proc_sketch_path']).convert('RGB')
        if self.img_tfm:    img    = self.img_tfm(img)
        if self.sketch_tfm: sketch = self.sketch_tfm(sketch)
        return {
            'sketch' : sketch,
            'image'  : img,
            'name'   : row['name'],
            'colour' : row['colour'],
            'idx'    : idx
        }

# rebuild splits using saved indices
splits = torch.load('/kaggle/working/split_indices.pt')
full_dataset = MyntraSketchDataset(df, img_tfm=img_transform, sketch_tfm=sketch_transform)

train_ds = Subset(full_dataset, splits['train_indices'])
val_ds   = Subset(full_dataset, splits['val_indices'])
test_ds  = Subset(full_dataset, splits['test_indices'])

PIN = torch.cuda.is_available()
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2, pin_memory=PIN)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2, pin_memory=PIN)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False, num_workers=2, pin_memory=PIN)

print(f"Train batches: {len(train_loader)}")
print(f"Val   batches: {len(val_loader)}")
print(f"Test  batches: {len(test_loader)}")



## 5. Sanity check with one real batch

In [10]:

batch = next(iter(train_loader))
sketches = batch['sketch']             # [32, 1, 224, 224]

with torch.no_grad():
    features = encoder(sketches)       # [32, 512, 7, 7]

print(f"\nInput  shape: {sketches.shape}")
print(f"Output shape: {features.shape}")
print(f"Expected    : torch.Size([32, 512, 7, 7])")
print(f"Match       : {features.shape == torch.Size([32, 512, 7, 7])}")

print('True')

NameError: name 'train_loader' is not defined